# 📘 Notebook 2: The Chain Rule & Backpropagation

**Week 2 — Calculus, Optimization & Probability Theory**

**Difficulty:** ⭐⭐⭐ (Intermediate)

---

## 🏠 Why Does This Matter?

Your house price model has **3 layers** now:

```
Input features → [Layer 1: linear] → [Layer 2: ReLU] → [Layer 3: linear] → Predicted Price → Loss
```

To improve the prediction, you need to adjust **every weight in every layer**.  
But Layer 1's weights are **far from the loss** — three steps away!

**How do you know how much to adjust Layer 1's weights?**

Answer: You trace the effect backwards:
> "How does Layer 1's output affect Layer 2? How does Layer 2 affect Layer 3? How does Layer 3 affect the loss?"

You **multiply the rates of change** along this chain.  
That multiplication rule is the **chain rule** — and applying it backwards through all layers is called **backpropagation**.

---

## 📚 Prerequisites
- Notebook 1 (partial derivatives, Jacobians)
- Basic matrix multiplication (Week 1)

---

## Part 1: The Chain Rule (Single Variable)

### Plain English First

Imagine this chain:
> Coffee → Work hours → Salary

- Each extra cup of coffee: +2 hours of work  
- Each extra hour of work: +\$50 salary

**Question:** If you drink one more cup of coffee, how much does your salary change?

**Answer:** 2 hours/cup × \$50/hour = **\$100/cup** — you multiply the rates!

That's the chain rule:

$$\frac{d(\text{salary})}{d(\text{coffee})} = \frac{d(\text{salary})}{d(\text{hours})} \times \frac{d(\text{hours})}{d(\text{coffee})}$$

### In Math Terms

If `y = f(g(x))` (a function of a function), then:

$$\frac{dy}{dx} = \frac{dy}{dg} \cdot \frac{dg}{dx} = f'(g(x)) \cdot g'(x)$$

### House Price Example

```
w (weight) → z = w·x + b (linear step) → h = ReLU(z) (activation) → L = (h - price)² (loss)
```

To find `dL/dw`, chain rule says:
$$\frac{dL}{dw} = \frac{dL}{dh} \cdot \frac{dh}{dz} \cdot \frac{dz}{dw}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────
# Verify the chain rule numerically
# Function: L(w) = (ReLU(w*x + b) - price)²
# where x=1500 sqft (normalized to 1.0), b=0.1, price=1.0 (normalized)
# ─────────────────────────────────────────────────────────────────

x_feat  = 1.0    # normalized square footage input
b_val   = 0.1    # bias
price   = 1.0    # normalized true price

def relu(z):
    """ReLU activation: pass positive values, set negatives to zero."""
    return np.maximum(0, z)

def full_pipeline(w):
    """
    Forward pass of a single-neuron house price model:
    w → z = w*x + b → h = ReLU(z) → L = (h - price)²
    """
    z = w * x_feat + b_val    # linear step
    h = relu(z)               # activation
    L = (h - price)**2        # squared error loss
    return L

def chain_rule_derivative(w):
    """
    dL/dw computed by chain rule:
      dL/dh = 2(h - price)
      dh/dz = 1 if z>0 else 0   (ReLU derivative)
      dz/dw = x_feat             (from z = w*x + b)
    """
    z      = w * x_feat + b_val     # step 1: linear
    h      = relu(z)                # step 2: activation
    dL_dh  = 2 * (h - price)        # derivative of loss w.r.t. h
    dh_dz  = 1.0 if z > 0 else 0.0  # ReLU derivative: 1 when active, 0 when off
    dz_dw  = x_feat                 # from z = w*x + b, dz/dw = x
    return dL_dh * dh_dz * dz_dw   # chain rule: multiply all three together

# Test at several weight values
print("Chain rule verification for: L = (ReLU(w·x + b) − price)²")
print(f"(x={x_feat}, b={b_val}, price={price})")
print()
print(f"{'w':8} | {'Chain rule dL/dw':20} | {'Numerical dL/dw':20} | {'Error':10}")
print("-" * 65)

eps = 1e-5
for w_test in [-1.0, -0.2, 0.0, 0.5, 1.0, 2.0]:
    analytic  = chain_rule_derivative(w_test)                               # our chain rule
    numerical = (full_pipeline(w_test+eps) - full_pipeline(w_test-eps))/(2*eps)  # numerical check
    error     = abs(analytic - numerical)
    print(f"{w_test:8.2f} | {analytic:20.10f} | {numerical:20.10f} | {error:.2e}")

## Part 2: Computational Graphs — Visualizing the Chain

### Plain English First

Modern ML frameworks (PyTorch, TensorFlow) draw every calculation as a **graph**:
- Each **node** is an operation (+, ×, ReLU, etc.)
- Each **arrow** carries a value from one operation to the next

**Forward pass:** values flow left → right (input to output)  
**Backward pass (backprop):** gradients flow right → left (loss back to weights)

### House Price Computational Graph

```
sqft ─────┐
           ├──[×w1]──> z1
beds ─────┘                 }
                    z1 + z2 + b ──> z_total ──> [ReLU] ──> h ──> [subtract price] ──> e ──> [square] ──> L
           ┌──[×w2]──> z2
```

During backprop, we compute `dL/dw1` and `dL/dw2` by tracing this graph backwards.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# MANUAL BACKPROPAGATION through a house price computational graph
#
# Model: predicted_price = ReLU(w1 * sqft + w2 * beds + b)
# Loss:  L = (predicted_price - true_price)²
#
# We will compute dL/dw1 and dL/dw2 by hand using the chain rule,
# then verify with finite differences.
# ─────────────────────────────────────────────────────────────────

# ─── A single training example ────────────────────────────────────
sqft       = 1.5    # normalized square footage
beds       = 0.8    # normalized bedroom count
true_price = 1.2    # normalized true price

# ─── Current weights (starting point before training) ─────────────
w1 = 0.5    # weight for sqft
w2 = 0.3    # weight for beds
b  = 0.1    # bias term

print("═" * 55)
print("STEP 1: FORWARD PASS (left → right)")
print("═" * 55)

# Compute each intermediate value and SAVE them — we need them in backprop!
z = w1 * sqft + w2 * beds + b    # linear combination
h = relu(z)                       # ReLU activation
e = h - true_price                # prediction error
L = e**2                          # squared loss

print(f"  z = w1*sqft + w2*beds + b")
print(f"    = {w1}×{sqft} + {w2}×{beds} + {b}")
print(f"    = {w1*sqft:.3f} + {w2*beds:.3f} + {b} = {z:.4f}")
print(f"  h = ReLU({z:.4f}) = {h:.4f}  (ReLU: max(0, z))")
print(f"  e = h - true_price = {h:.4f} - {true_price} = {e:.4f}")
print(f"  L = e² = {e:.4f}² = {L:.4f}")

print()
print("═" * 55)
print("STEP 2: BACKWARD PASS (right → left, chain rule)")
print("═" * 55)

# Start from the loss and work backwards
# Each step: multiply the incoming gradient by the local derivative

dL_dL = 1.0                    # seed: gradient of L w.r.t. itself = 1

# Node: L = e²  →  dL/de = 2e
dL_de = 2 * e * dL_dL          # chain: dL/de = (dL/dL) * (d(e²)/de)

# Node: e = h - true_price  →  de/dh = 1
dL_dh = 1.0 * dL_de            # chain: dL/dh = dL/de * de/dh = dL/de * 1

# Node: h = ReLU(z)  →  dh/dz = 1 if z>0 else 0
dh_dz = 1.0 if z > 0 else 0.0  # ReLU passes gradient through only when active
dL_dz = dh_dz * dL_dh          # chain: dL/dz = dL/dh * dh/dz

# Node: z = w1*sqft + w2*beds + b
#   dz/dw1 = sqft,  dz/dw2 = beds,  dz/db = 1
dL_dw1 = sqft * dL_dz           # chain: dL/dw1 = dL/dz * dz/dw1
dL_dw2 = beds * dL_dz           # chain: dL/dw2 = dL/dz * dz/dw2
dL_db  = 1.0  * dL_dz           # chain: dL/db  = dL/dz * dz/db

print(f"  dL/dL  = {dL_dL}                         (seed)")
print(f"  dL/de  = 2e = 2×{e:.4f} = {dL_de:.4f}   (from L=e²)")
print(f"  dL/dh  = dL/de × 1 = {dL_dh:.4f}         (from e=h-price, de/dh=1)")
print(f"  dh/dz  = {dh_dz}  (ReLU: 1 since z={z:.4f} > 0)")
print(f"  dL/dz  = dL/dh × dh/dz = {dL_dz:.4f}")
print(f"  dL/dw1 = dL/dz × sqft = {dL_dz:.4f} × {sqft} = {dL_dw1:.4f}  ← adjust w1 by this")
print(f"  dL/dw2 = dL/dz × beds = {dL_dz:.4f} × {beds} = {dL_dw2:.4f}  ← adjust w2 by this")
print(f"  dL/db  = dL/dz × 1    = {dL_db:.4f}              ← adjust b  by this")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# VERIFY our manual backprop with finite differences
# ─────────────────────────────────────────────────────────────────

def forward(w1, w2, b):
    """Full forward pass: returns loss L given weights w1, w2, b."""
    z = w1 * sqft + w2 * beds + b
    h = relu(z)
    e = h - true_price
    return e**2

eps = 1e-5
# Numerical gradients: nudge each parameter independently
dL_dw1_num = (forward(w1+eps, w2, b) - forward(w1-eps, w2, b)) / (2*eps)
dL_dw2_num = (forward(w1, w2+eps, b) - forward(w1, w2-eps, b)) / (2*eps)
dL_db_num  = (forward(w1, w2, b+eps) - forward(w1, w2, b-eps)) / (2*eps)

print("\nGRADIENT CHECK (manual backprop vs finite differences):")
print(f"{'Gradient':10} | {'Manual backprop':18} | {'Finite diff':18} | {'Error':10}")
print("-" * 65)
for name, manual, numerical in [
    ('dL/dw1', dL_dw1, dL_dw1_num),
    ('dL/dw2', dL_dw2, dL_dw2_num),
    ('dL/db',  dL_db,  dL_db_num),
]:
    print(f"{name:10} | {manual:18.10f} | {numerical:18.10f} | {abs(manual-numerical):.2e}")

print()
print("Errors are ~1e-10 → our manual backprop is correct!")

## Part 3: Backpropagation Through a Full 2-Layer Network

### Plain English First

Now let's scale up to a real network with **two weight matrices**: `W1` and `W2`.  
The house price model is now:

```
features (2D) → [W1, b1] → [ReLU] → hidden (4D) → [W2, b2] → predicted price (1D)
                                                                      ↓
                                                               MSE Loss
```

Backprop works the same way — just more steps in the chain.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 2-Layer Neural Network for House Price Prediction
#
# Architecture:
#   Input:   2 features (sqft_norm, beds_norm)
#   Layer 1: 2 → 4 units (W1 is 4×2, b1 is 4)
#   ReLU activation
#   Layer 2: 4 → 1 unit  (W2 is 1×4, b2 is 1)
#   Output:  1 (predicted price)
#   Loss:    MSE = 0.5 * (predicted - true)²  (0.5 for cleaner gradient)
# ─────────────────────────────────────────────────────────────────

np.random.seed(42)    # for reproducibility

# Random weight initialization (small values prevent large initial loss)
W1 = np.random.randn(4, 2) * 0.1   # shape: (4 hidden, 2 inputs)
b1 = np.zeros(4)                    # shape: (4,)
W2 = np.random.randn(1, 4) * 0.1   # shape: (1 output, 4 hidden)
b2 = np.zeros(1)                    # shape: (1,)

# One training example: house with sqft_norm=1.5, beds_norm=0.8
x_input    = np.array([1.5, 0.8])   # feature vector, shape: (2,)
y_true     = np.array([1.2])        # true price, shape: (1,)

# ═══════════════════════════════════════
# FORWARD PASS
# ═══════════════════════════════════════

# Layer 1: linear transformation
z1   = W1 @ x_input + b1    # shape: (4,)   matrix-vector multiply

# Layer 1: activation (ReLU)
h1   = np.maximum(0, z1)    # shape: (4,)   element-wise max with 0

# Layer 2: linear transformation
z2   = W2 @ h1 + b2         # shape: (1,)   produces predicted price

# Loss: 0.5 * MSE (the 0.5 makes the gradient cleaner: 2*0.5=1)
loss = 0.5 * np.sum((z2 - y_true)**2)  # scalar

print("FORWARD PASS:")
print(f"  x (input features)  = {x_input}")
print(f"  z1 = W1@x + b1      = {z1}")
print(f"  h1 = ReLU(z1)       = {h1}")
print(f"  z2 = W2@h1 + b2     = {z2}   ← predicted price")
print(f"  Loss = 0.5*(z2-y)²  = {loss:.6f}")

# ═══════════════════════════════════════
# BACKWARD PASS (chain rule)
# ═══════════════════════════════════════

# Start from the loss, work backwards layer by layer

# Gradient of loss w.r.t. z2
#   L = 0.5*(z2-y)²  →  dL/dz2 = (z2-y)
dL_dz2 = z2 - y_true                    # shape: (1,)

# Gradient w.r.t. W2 and b2 (Layer 2 weights)
#   z2 = W2@h1 + b2  →  dz2/dW2 = h1 (outer product)
#                     →  dz2/db2 = 1
dL_dW2 = np.outer(dL_dz2, h1)           # shape: (1, 4) — outer product
dL_db2 = dL_dz2                         # shape: (1,)

# Gradient flowing back through W2 to h1
#   dL/dh1 = W2.T @ dL/dz2
dL_dh1 = W2.T @ dL_dz2                  # shape: (4,)

# Gradient through ReLU: passes through where z1 > 0, blocks where z1 ≤ 0
#   dh1/dz1 = 1 if z1 > 0 else 0
dL_dz1 = dL_dh1 * (z1 > 0).astype(float)  # shape: (4,)  element-wise gate

# Gradient w.r.t. W1 and b1 (Layer 1 weights)
#   z1 = W1@x + b1  →  dz1/dW1 = x (outer product)
dL_dW1 = np.outer(dL_dz1, x_input)     # shape: (4, 2)
dL_db1 = dL_dz1                        # shape: (4,)

print("\nBACKWARD PASS (gradients for each parameter):")
print(f"  dL/dz2 = {dL_dz2}")
print(f"  dL/dW2 = {dL_dW2}")
print(f"  dL/db2 = {dL_db2}")
print(f"  dL/dh1 = {dL_dh1}")
print(f"  dL/dz1 = {dL_dz1}  (after ReLU gate)")
print(f"  dL/dW1 =\n{dL_dW1}")
print(f"  dL/db1 = {dL_db1}")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# VERIFY all gradients with finite differences
# This is the "gradient check" step done in every serious ML framework
# ─────────────────────────────────────────────────────────────────

def full_forward(W1, b1, W2, b2, x, y):
    """Complete forward pass. Returns the scalar loss."""
    z1   = W1 @ x + b1
    h1   = np.maximum(0, z1)
    z2   = W2 @ h1 + b2
    loss = 0.5 * np.sum((z2 - y)**2)
    return loss

eps = 1e-5

# Numerical gradient for W1 (nudge each element of W1 one at a time)
dL_dW1_num = np.zeros_like(W1)
for i in range(W1.shape[0]):        # loop over rows
    for j in range(W1.shape[1]):    # loop over columns
        W1p = W1.copy(); W1p[i,j] += eps    # nudge up
        W1m = W1.copy(); W1m[i,j] -= eps    # nudge down
        dL_dW1_num[i,j] = (full_forward(W1p,b1,W2,b2,x_input,y_true) -
                            full_forward(W1m,b1,W2,b2,x_input,y_true)) / (2*eps)

# Numerical gradient for W2
dL_dW2_num = np.zeros_like(W2)
for i in range(W2.shape[0]):
    for j in range(W2.shape[1]):
        W2p = W2.copy(); W2p[i,j] += eps
        W2m = W2.copy(); W2m[i,j] -= eps
        dL_dW2_num[i,j] = (full_forward(W1,b1,W2p,b2,x_input,y_true) -
                            full_forward(W1,b1,W2m,b2,x_input,y_true)) / (2*eps)

print("GRADIENT CHECK:")
print(f"  dL/dW1 max error: {np.max(np.abs(dL_dW1 - dL_dW1_num)):.2e}  ← should be < 1e-6")
print(f"  dL/dW2 max error: {np.max(np.abs(dL_dW2 - dL_dW2_num)):.2e}  ← should be < 1e-6")
print()
print("Both pass! Our manual backprop is computing the correct gradients.")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# TRAIN the 2-layer house price network for 500 steps
# using our manual backprop
# ─────────────────────────────────────────────────────────────────

np.random.seed(42)

# Re-initialize weights
W1 = np.random.randn(4, 2) * 0.1
b1 = np.zeros(4)
W2 = np.random.randn(1, 4) * 0.1
b2 = np.zeros(1)

lr          = 0.02     # learning rate: how large a step to take each update
n_steps     = 500      # number of training iterations
loss_history = []      # store loss at each step to plot convergence

for step in range(n_steps):
    # ── Forward pass ──────────────────────────────────────────────
    z1   = W1 @ x_input + b1
    h1   = np.maximum(0, z1)
    z2   = W2 @ h1 + b2
    loss = 0.5 * np.sum((z2 - y_true)**2)
    loss_history.append(float(loss))

    # ── Backward pass (chain rule) ─────────────────────────────────
    dL_dz2 = z2 - y_true
    dL_dW2 = np.outer(dL_dz2, h1)
    dL_db2 = dL_dz2
    dL_dh1 = W2.T @ dL_dz2
    dL_dz1 = dL_dh1 * (z1 > 0).astype(float)
    dL_dW1 = np.outer(dL_dz1, x_input)
    dL_db1 = dL_dz1

    # ── Parameter update (gradient descent) ───────────────────────
    W2 -= lr * dL_dW2    # move W2 against the gradient
    b2 -= lr * dL_db2
    W1 -= lr * dL_dW1    # move W1 against the gradient (many layers back!)
    b1 -= lr * dL_db1

# ── Plot the loss curve ───────────────────────────────────────────
plt.figure(figsize=(10, 4))
plt.plot(loss_history, 'steelblue', linewidth=2)
plt.yscale('log')
plt.xlabel('Training step', fontsize=12)
plt.ylabel('Loss (log scale)', fontsize=12)
plt.title('2-Layer House Price Network: Loss Decreases via Backprop + Gradient Descent', fontsize=13)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Initial loss:  {loss_history[0]:.6f}")
print(f"Final loss:    {loss_history[-1]:.8f}")
print(f"Final prediction: {z2[0]:.4f}   (true price: {y_true[0]})")
print()
print("The chain rule allowed gradients to flow back through 2 layers,")
print("enabling us to correctly adjust W1 even though it is far from the loss.")

---

## ✅ Self-Check Questions

**1.** In the chain `w → z → h → L`, the chain rule says `dL/dw = ?`  
   Write it out as a product of three local derivatives.

**2.** What does the ReLU "gate" do to gradients during backprop?  
   What happens to a gradient when it passes through a neuron that was "off" (z < 0)?

**3.** Why do we compute the forward pass before the backward pass?  
   (Hint: what values do we use during backprop that we computed in the forward pass?)

**4.** If `dL/dW2` is large and positive, does that mean we should increase or decrease `W2`?  
   Write the gradient descent update rule.

**5.** Our `dL/dW1 = np.outer(dL_dz1, x_input)`. Why is it an outer product of these two vectors?  
   (Hint: `z1 = W1 @ x`. Each element of `z1` depends on one row of `W1` and all of `x`.)

<details>
<summary>Click to see answers</summary>

1. `dL/dw = (dL/dh) × (dh/dz) × (dz/dw)` — multiply local derivatives along the chain.

2. ReLU **blocks** the gradient when z < 0 (multiplies by 0), and **passes** it unchanged when z > 0 (multiplies by 1). This is the "dying ReLU" problem in deep networks.

3. We need intermediate values like `z1`, `h1`, `z2` that were computed in the forward pass. Without them we can't compute the local derivatives during backprop.

4. **Decrease** W2. The update rule is: `W2 ← W2 - lr × dL/dW2`.

5. Because `W1[i,j]` only affects `z1[i]`, and `z1[i] = W1[i,:] @ x`. So `dz1[i]/dW1[i,j] = x[j]`. Stacking all these gives the outer product `dL_dz1 ⊗ x_input`.

</details>

---

## 📌 Summary

| Concept | One-line meaning | House price example |
|---------|-----------------|---------------------|
| **Chain rule** | Derivative of composed functions = product of local derivatives | How w1 in Layer 1 affects the final loss |
| **Forward pass** | Compute all intermediate values L→R | Calculate predicted price from features |
| **Backward pass** | Compute all gradients R→L using chain rule | Figure out how to fix each weight |
| **Gradient check** | Verify backprop with finite differences | Sanity check: is our backprop correct? |
| **ReLU gate** | Passes gradients when active, blocks when not | Some neurons don't contribute to learning |

**➡️ Next Notebook:** Gradient Vectors — the multi-dimensional direction that tells us which way to move all weights simultaneously.